In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    "./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()

        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(),
            nn.Conv2d(
                in_channels,
                growth_rate,
                kernel_size=3,
                padding=1
            )
        )

    def forward(self, x):
        new_features = self.layer(x)
        return torch.cat([x, new_features], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_channels, num_layers, growth_rate):
        super().__init__()

        layers = []
        channels = in_channels

        for _ in range(num_layers):
            layers.append(
                DenseLayer(channels, growth_rate)
            )
            channels += growth_rate

        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)


class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(),
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1
            ),
            nn.AvgPool2d(
                kernel_size=2,
                stride=2
            )
        )

    def forward(self, x):
        return self.layer(x)


class DenseNet121(nn.Module):
    def __init__(self, num_classes=10, growth_rate=32):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                1,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.block1 = DenseBlock(64, 6, growth_rate)

        self.transition1 = TransitionLayer(
            self.block1.out_channels,
            self.block1.out_channels // 2
        )

        self.block2 = DenseBlock(
            self.block1.out_channels // 2,
            12,
            growth_rate
        )

        self.transition2 = TransitionLayer(
            self.block2.out_channels,
            self.block2.out_channels // 2
        )

        self.block3 = DenseBlock(
            self.block2.out_channels // 2,
            24,
            growth_rate
        )

        self.transition3 = TransitionLayer(
            self.block3.out_channels,
            self.block3.out_channels // 2
        )

        self.block4 = DenseBlock(
            self.block3.out_channels // 2,
            16,
            growth_rate
        )

        self.bn = nn.BatchNorm2d(
            self.block4.out_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(
            self.block4.out_channels,
            num_classes
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.block1(x)
        x = self.transition1(x)

        x = self.block2(x)
        x = self.transition2(x)

        x = self.block3(x)
        x = self.transition3(x)

        x = self.block4(x)

        x = self.bn(x)
        x = self.relu(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        return self.fc(x)


model = DenseNet121().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {total_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")